# NoLemming: TRIBE v2 Brain Encoder

This notebook runs Meta's TRIBE v2 brain encoding model on a free Colab GPU.
It takes your stimulus files, encodes them as neural responses, and exports
`.npy` files you can use with NoLemming locally.

**Requirements:**
- Google Colab with GPU runtime (Runtime → Change runtime type → T4 GPU)
- HuggingFace token with LLaMA 3.2 access

**Time:** ~5 min setup, ~30s per stimulus file

In [ ]:
# Step 1: Install TRIBE v2
!pip install -q git+https://github.com/facebookresearch/tribev2.git
!pip install -q huggingface_hub

In [ ]:
# Step 2: Authenticate with HuggingFace
# You need access to meta-llama/Llama-3.2-1B
from huggingface_hub import login

# Paste your HuggingFace token here
HF_TOKEN = ""  # <-- PASTE YOUR TOKEN

login(token=HF_TOKEN)

In [ ]:
# Step 3: Load TRIBE v2 model
from tribev2 import TribeModel
from pathlib import Path
import numpy as np

cache_folder = Path("./cache")
cache_folder.mkdir(exist_ok=True)

print("Loading TRIBE v2 model (first run downloads ~1GB)...")
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder=cache_folder)
print("Model loaded!")

In [ ]:
# Step 4: Upload your stimulus files
# Option A: Upload from local machine
from google.colab import files
uploaded = files.upload()
stimulus_files = list(uploaded.keys())
print(f"Uploaded: {stimulus_files}")

In [ ]:
# Option B: Use sample stimuli (skip Step 4 if using this)
# Uncomment to create a sample text stimulus:
#
# sample_text = """Apple reports Q4 earnings. Revenue $143.8B beats estimates.
# EPS $2.84 vs $2.66 expected. Record quarter with strong iPhone and
# Services growth. AI features highlighted but no major announcements."""
#
# Path("sample_earnings.txt").write_text(sample_text)
# stimulus_files = ["sample_earnings.txt"]

In [ ]:
# Step 5: Encode each stimulus
output_dir = Path("./neural_responses")
output_dir.mkdir(exist_ok=True)

for stim_file in stimulus_files:
    stim_path = Path(stim_file)
    ext = stim_path.suffix.lower()
    print(f"\nEncoding: {stim_file}")

    # Dispatch based on file type
    if ext in (".txt", ".md"):
        df = model.get_events_dataframe(text_path=str(stim_path))
    elif ext in (".wav", ".mp3", ".flac", ".ogg"):
        df = model.get_events_dataframe(audio_path=str(stim_path))
    elif ext in (".mp4", ".avi", ".mkv", ".mov", ".webm"):
        df = model.get_events_dataframe(video_path=str(stim_path))
    else:
        print(f"  Skipping unsupported format: {ext}")
        continue

    # Run inference
    preds, segments = model.predict(events=df)
    print(f"  Shape: {preds.shape} (timesteps, vertices)")
    print(f"  Mean activation: {np.mean(preds):.4f}")
    print(f"  Max activation: {np.max(preds):.4f}")

    # Save as .npy
    out_path = output_dir / f"{stim_path.stem}_neural.npy"
    np.save(out_path, preds.astype(np.float32))
    print(f"  Saved: {out_path}")

print(f"\nAll responses saved to {output_dir}/")

In [ ]:
# Step 6: Download the neural response files
import shutil

# Zip all responses
shutil.make_archive("neural_responses", "zip", output_dir)
files.download("neural_responses.zip")
print("Downloaded! Unzip and place .npy files in your NoLemming project.")

## Using the .npy files with NoLemming

```python
import numpy as np
from nolemming.core.types import NeuralResponse

# Load the pre-computed neural response
activations = np.load("aapl_q4_2025_neural.npy")
response = NeuralResponse(activations=activations)

# Use directly in the pipeline
from nolemming.mapping.compressor import VoxelCompressor
from nolemming.mapping.archetypes import ArchetypeClusterer
from nolemming.mapping.brain_atlas import BrainAtlas

compressor = VoxelCompressor(n_dims=64)
compressor.fit_single(response)
compressed = compressor.compress_timesteps(response)

atlas = BrainAtlas()
archetypes = ArchetypeClusterer(n_archetypes=8).cluster(
    compressed, atlas, response.activations
)
# ... continue with simulation
```